In [ ]:

# WT LFP PROCESSING — Initialization

%pip -q install -U ONE-api ibllib iblatlas scipy

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from one.api import ONE
from iblatlas.regions import BrainRegions

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    username="intbrainlab",
    password="international",
    silent=True
)

br = BrainRegions()

ROOT = Path(
    "/content/drive/MyDrive/Reanalysis/"
    "Our_Analysis/lfp/spontaneous/WT/processing"
)
ROOT.mkdir(parents=True, exist_ok=True)

inventory = pd.read_csv(ROOT / "WT_LFP_inventory.csv")
inventory = inventory[
    inventory["lf_available"].astype(str).str.lower() == "true"
].drop_duplicates("pid").reset_index(drop=True)

# Analysis parameters
N_WINDOWS = 30
WINDOW_S = 2.0
GUARD_S = 0.5
TUKEY_K = 3.0
MIN_ANIMALS = 5

BANDS = {
    "Delta": (1, 4),
    "Theta": (4, 8),
    "Alpha-like": (8, 12),
    "Beta": (12, 30),
    "Low gamma": (30, 55),
    "High gamma": (65, 100)
}

REGIONS = [
    "DG", "CA1", "CA3", "SUB",
    "VISp", "MOs", "CP", "MRN", "APN"
]

print("WT animals:", inventory["subject"].nunique())
print("Sessions:", inventory["eid"].nunique())
print("Insertions:", len(inventory))

# Imports required by later cells (loaded once).
from collections import defaultdict
from scipy.signal import welch
from brainbox.io.one import SpikeSortingLoader


In [ ]:

# FIX — Recover anatomy from IBL collections and revisions
# Overrides the old get_channel_groups() function.


br = BrainRegions()

def get_channel_groups(eid, probe):
    """
    Discover anatomical files in:
      alf/probeXX/
      alf/probeXX/pykilosort/
      alf/probeXX/#revision#/
      alf/probeXX/pykilosort/#revision#/

    Never pair arrays from different revisions.
    Map electrodeSites through channel coordinates when needed.
    """

    paths = [str(p) for p in one.list_datasets(eid)]
    prefix = f"alf/{probe}/"

    catalog = defaultdict(set)

    for path in paths:
        if not path.startswith(prefix):
            continue

        parts = path.split("/")
        filename = parts[-1]

        if not filename.startswith(
            ("channels.", "electrodeSites.")
        ):
            continue

        collection = "/".join(parts[:-1])
        revision = None

        # ONE revision folders have the form #YYYY-MM-DD#.
        if parts[-2].startswith("#") and parts[-2].endswith("#"):
            revision = parts[-2][1:-1]
            collection = "/".join(parts[:-2])

        object_name = filename.split(".")[0]
        catalog[(collection, revision, object_name)].add(filename)

    def load_attr(collection, revision, obj, attr):
        kwargs = {"collection": collection}
        if revision:
            kwargs["revision"] = revision

        return np.asarray(one.load_dataset(
            eid,
            f"{obj}.{attr}.npy",
            **kwargs
        ))

    # Gather candidates with atlas annotations.
    candidates = []

    for (collection, revision, obj), filenames in catalog.items():
        atlas_name = f"{obj}.brainLocationIds_ccf_2017.npy"

        if atlas_name not in filenames:
            continue

        candidates.append({
            "collection": collection,
            "revision": revision,
            "object": obj,
            "filenames": filenames
        })

    # Prefer channel mappings with explicit raw indices.
    # Evaluate every candidate, not only the first.
    candidates.sort(
        key=lambda x: (
            x["object"] != "channels",
            x["revision"] is None
        )
    )

    recovered = []
    failures = []

    for candidate in candidates:
        col = candidate["collection"]
        rev = candidate["revision"]
        obj = candidate["object"]
        names = candidate["filenames"]

        try:
            atlas = load_attr(
                col, rev, obj,
                "brainLocationIds_ccf_2017"
            ).ravel().astype(int)

            raw = None
            mapping = None

            # Preferred: explicit channel-to-raw mapping.
            if f"{obj}.rawInd.npy" in names:
                raw = load_attr(
                    col, rev, obj, "rawInd"
                ).ravel().astype(int)

                if len(raw) != len(atlas):
                    raise ValueError(
                        "Atlas/rawInd length mismatch"
                    )

                mapping = "explicit_rawInd"

            # Alternative: match electrode-site coordinates
            # against channels with known raw indices.
            elif obj == "electrodeSites" and (
                "electrodeSites.localCoordinates.npy" in names
            ):
                site_xy = load_attr(
                    col, rev, obj, "localCoordinates"
                )

                for (other_col, other_rev, other_obj), other_names in catalog.items():
                    if other_obj != "channels":
                        continue

                    required = {
                        "channels.rawInd.npy",
                        "channels.localCoordinates.npy"
                    }

                    if not required.issubset(other_names):
                        continue

                    try:
                        ch_raw = load_attr(
                            other_col, other_rev,
                            "channels", "rawInd"
                        ).ravel().astype(int)

                        ch_xy = load_attr(
                            other_col, other_rev,
                            "channels", "localCoordinates"
                        )

                        if len(ch_raw) != len(ch_xy):
                            continue

                        def coord_key(x):
                            return tuple(np.round(x[:2], 3))

                        coordinate_to_raw = {}

                        for xy, channel in zip(ch_xy, ch_raw):
                            key = coord_key(xy)
                            coordinate_to_raw.setdefault(
                                key, []
                            ).append(int(channel))

                        match_raw = []
                        match_atlas = []

                        for xy, atlas_id in zip(site_xy, atlas):
                            matches = coordinate_to_raw.get(
                                coord_key(xy), []
                            )

                            if len(matches) == 1:
                                match_raw.append(matches[0])
                                match_atlas.append(atlas_id)

                        if match_raw:
                            raw = np.asarray(
                                match_raw, dtype=int
                            )
                            atlas = np.asarray(
                                match_atlas, dtype=int
                            )
                            mapping = "coordinate_match"
                            break

                    except Exception:
                        continue

            if raw is None:
                raise ValueError(
                    "No verified raw-channel mapping"
                )

            if len(np.unique(raw)) != len(raw):
                raise ValueError("Duplicate raw indices")

            valid = (
                (raw >= 0) &
                (raw < 384)
            )

            raw = raw[valid]
            atlas = atlas[valid]

            acronyms = br.id2acronym(atlas)

            groups = defaultdict(list)

            for channel, acronym in zip(raw, acronyms):
                region, subregion = classify_region(
                    str(acronym)
                )

                if region is None:
                    continue

                groups[(region, "ALL")].append(
                    int(channel)
                )
                groups[(region, subregion)].append(
                    int(channel)
                )

            if not groups:
                raise ValueError(
                    "No target regions in annotation"
                )

            groups = {
                key: np.unique(values).astype(int)
                for key, values in groups.items()
            }

            n_contacts = len(set(
                ch
                for values in groups.values()
                for ch in values
            ))

            recovered.append({
                "groups": groups,
                "n_contacts": n_contacts,
                "collection": col,
                "revision": rev,
                "mapping": mapping
            })

        except Exception as exc:
            failures.append(
                f"{obj} {col} {rev}: {exc}"
            )

    if not recovered:
        raise RuntimeError(
            "No usable anatomical mapping. "
            + "; ".join(failures[:5])
        )

    # Choose the mapping covering the most target contacts.
    best = max(
        recovered,
        key=lambda x: x["n_contacts"]
    )

    return best["groups"]

print("Updated anatomical loader installed.")


In [ ]:

# WT LFP PROCESSING — Full WT reprocessing


INSERTION_DIR = ROOT / "per_insertion"
INSERTION_DIR.mkdir(parents=True, exist_ok=True)

log_rows = []
session_window_cache = {}

for number, row in enumerate(inventory.itertuples(index=False), 1):

    eid_i = str(row.eid)
    pid_i = str(row.pid)
    subject = str(row.subject)
    probe = str(row.probe)

    output_path = INSERTION_DIR / f"{pid_i}.csv"

    print(
        f"[{number}/{len(inventory)}] "
        f"{subject} — {probe}",
        flush=True
    )

    try:
        # 1. Recover anatomy from all available collections.
        groups = get_channel_groups(eid_i, probe)

        # 2. Select quiet windows once per session.
        if eid_i not in session_window_cache:
            session_window_cache[eid_i] = get_quiet_windows(
                eid_i
            )

        windows, n_eligible = session_window_cache[eid_i]

        if len(windows) == 0:
            raise ValueError("No eligible quiet windows")

        # 3. Initialize remote LF stream.
        ssl_i = SpikeSortingLoader(
            pid=pid_i,
            one=one
        )

        reader = ssl_i.raw_electrophysiology(
            band="lf",
            stream=True
        )

        fs_i = float(reader.fs)
        n_neural = int(reader.nc - reader.nsync)
        n_samples = int(round(WINDOW_S * fs_i))

        # 4. Restrict analysis to anatomically labeled contacts.
        selected_channels = np.unique(
            np.concatenate(list(groups.values()))
        )

        selected_channels = selected_channels[
            (selected_channels >= 0) &
            (selected_channels < n_neural)
        ]

        if len(selected_channels) == 0:
            raise ValueError("No usable neural channels")

        local_lookup = {
            int(ch): i
            for i, ch in enumerate(selected_channels)
        }

        group_indices = {}

        for key, raw_channels in groups.items():
            indices = [
                local_lookup[int(ch)]
                for ch in raw_channels
                if int(ch) in local_lookup
            ]

            if indices:
                group_indices[key] = np.asarray(
                    indices, dtype=int
                )

        # 5. Stream matched 2-second windows.
        spectra = []
        rms_values = []
        loaded_windows = []

        window_errors = []

        for t0, t1 in windows:
            try:
                probe_t0 = float(
                    ssl_i.timesprobe2times(
                        float(t0),
                        direction="reverse"
                    )
                )

                s0 = int(round(probe_t0 * fs_i))
                s1 = s0 + n_samples

                if s0 < 0:
                    continue

                block = np.asarray(
                    reader[s0:s1, :n_neural]
                )

                if block.shape != (n_samples, n_neural):
                    continue

                signal_uv = (
                    block[:, selected_channels]
                    .astype(np.float64) * 1e6
                )

                if not np.all(np.isfinite(signal_uv)):
                    continue

                frequencies, psd = welch(
                    signal_uv,
                    fs=fs_i,
                    window="hann",
                    nperseg=n_samples,
                    noverlap=0,
                    detrend="linear",
                    scaling="density",
                    axis=0
                )

                freq_mask = frequencies <= 100

                spectra.append(
                    psd[freq_mask, :].T.astype(np.float32)
                )

                rms_values.append(
                    np.std(signal_uv, axis=0)
                )

                loaded_windows.append((t0, t1))

            except Exception as exc:
                window_errors.append(str(exc)[:150])

        if not spectra:
            raise ValueError(
                "No LFP windows successfully loaded. "
                + "; ".join(window_errors[:3])
            )

        # Shape: windows x channels x frequencies
        psd_array = np.stack(spectra, axis=0)
        rms_array = np.stack(rms_values, axis=0)
        frequencies = frequencies[freq_mask]

        # 6. Calculate band power per anatomical group.
        results = []

        for (region, subregion), channel_idx in group_indices.items():

            metrics = calculate_group_power(
                psd_array,
                rms_array,
                frequencies,
                channel_idx
            )

            if metrics is None:
                continue

            results.append({
                "subject": subject,
                "genotype": "WT",
                "eid": eid_i,
                "pid": pid_i,
                "probe": probe,
                "region": region,
                "subregion": subregion,
                **metrics
            })

        if not results:
            raise ValueError("No regional power estimates")

        # 7. Overwrite old results after successful processing.
        pd.DataFrame(results).to_csv(
            output_path,
            index=False
        )

        pd.DataFrame(
            loaded_windows,
            columns=["start_s", "end_s"]
        ).to_csv(
            INSERTION_DIR / f"{pid_i}_windows.csv",
            index=False
        )

        log_rows.append({
            "pid": pid_i,
            "subject": subject,
            "status": "processed",
            "n_windows": len(loaded_windows),
            "n_regions": len(results),
            "error": ""
        })

        print(
            f"  OK | {len(loaded_windows)} windows | "
            f"{len(results)} anatomical groups"
        )

    except Exception as exc:
        # Remove any stale result from the previous run.
        if output_path.exists():
            output_path.unlink()

        log_rows.append({
            "pid": pid_i,
            "subject": subject,
            "status": "failed",
            "n_windows": 0,
            "n_regions": 0,
            "error": str(exc)
        })

        print(f"  FAILED: {str(exc)[:180]}")

    # Save progress after every insertion.
    pd.DataFrame(log_rows).to_csv(
        ROOT / "WT_LFP_processing_log.csv",
        index=False
    )

print("\nFull WT reprocessing finished.")

log_df = pd.DataFrame(log_rows)

print("Successful insertions:",
      (log_df["status"] == "processed").sum())

print("Failed insertions:",
      (log_df["status"] == "failed").sum())

print("Animals with successful insertions:",
      log_df.loc[
          log_df["status"] == "processed",
          "subject"
      ].nunique())


In [ ]:

# WT LFP PROCESSING — Final animal-level tables

BAND_COLUMNS = [
    f"{name}_{metric}"
    for name in BANDS
    for metric in ["absolute_uv2", "relative_pct"]
]

# Read only completed insertion-level result files.
files = list(INSERTION_DIR.glob("*.csv"))
files = [
    p for p in files
    if not p.name.endswith("_windows.csv")
]

if not files:
    raise RuntimeError("No completed insertion results found.")

insertion_results = pd.concat(
    [pd.read_csv(p) for p in files],
    ignore_index=True
)

insertion_results.to_csv(
    ROOT / "WT_LFP_insertion_results.csv",
    index=False
)

# Aggregate repeated sessions/probes within each animal.
# Each insertion contributes one estimate per anatomical group.
animal_results = (
    insertion_results
    .groupby(
        ["subject", "genotype", "region", "subregion"],
        as_index=False
    )
    .agg(
        **{
            col: (col, "median")
            for col in BAND_COLUMNS
        },
        n_sessions=("eid", "nunique"),
        n_insertions=("pid", "nunique"),
        total_channels=("n_channels", "sum")
    )
)

# Determine independent animal coverage.
coverage = (
    animal_results
    .groupby(["region", "subregion"], as_index=False)
    .agg(n_animals=("subject", "nunique"))
)

coverage["status"] = np.where(
    coverage["n_animals"] >= MIN_ANIMALS,
    "INCLUDED",
    "INSUFFICIENT_DATA"
)

# Apply the minimum-five-animals rule.
eligible = coverage.loc[
    coverage["status"] == "INCLUDED",
    ["region", "subregion"]
]

final_results = animal_results.merge(
    eligible,
    on=["region", "subregion"],
    how="inner"
)

# Save all tables.
animal_results.to_csv(
    ROOT / "WT_LFP_all_animal_results.csv",
    index=False
)

coverage.to_csv(
    ROOT / "WT_LFP_region_eligibility.csv",
    index=False
)

final_results.to_csv(
    ROOT / "WT_LFP_final_animal_results.csv",
    index=False
)

# Create a group-level summary table.
summary = (
    final_results
    .groupby(["region", "subregion"], as_index=False)
    .agg(
        n_animals=("subject", "nunique"),
        **{
            col: (col, "median")
            for col in BAND_COLUMNS
        }
    )
)

summary.to_csv(
    ROOT / "WT_LFP_group_summary.csv",
    index=False
)

print("Animals analyzed:", animal_results["subject"].nunique())
print("Eligible region/subregion groups:", len(eligible))
print("Final animal-level rows:", len(final_results))

print("\nRegion coverage:")
display(coverage.sort_values(
    ["region", "subregion"]
))

print("\nFinal group summary:")
display(summary)

print("\nFiles saved to:", ROOT)
